# Stage11: Evaluation & Risk Communication

Bootstrap uncertainty, cutoff scenarios, volatility-regime subgroups, and a stakeholder-ready conclusion for the SPY high-volatility baseline.


In [1]:
from pathlib import Path
import os,sys
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np,pandas as pd
from sklearn.metrics import average_precision_score
if Path.cwd().name=="homework11": os.chdir("..")
ROOT=Path.cwd()
if not (ROOT/"homework"/"homework11").is_dir():
    for x in (ROOT,*ROOT.parents):
        if (x/"homework"/"homework11").is_dir(): ROOT=x;break
H=ROOT/"homework"/"homework11";sys.path.insert(0,str(H))
from src.modeling import evaluate,run_baseline
from src.evaluation import bootstrap_metric
source=pd.read_csv(H/"data/raw/spy_feature_candidates_stage09_snapshot.csv",parse_dates=["date"])
r=run_baseline(source); y=r["test"]["label"].to_numpy(); p=r["test_probability"]


## Bootstrap PR-AUC uncertainty


In [2]:
boot=bootstrap_metric(y,p,average_precision_score,n_boot=600,seed=111)
ci=np.percentile(boot,[2.5,97.5]); print({"test_pr_auc":average_precision_score(y,p),"bootstrap_95_ci":ci.tolist()})


{'test_pr_auc': 0.2892991887266384, 'bootstrap_95_ci': [0.1750353999209764, 0.4631493486523629]}


## Scenario and subgroup comparisons


In [3]:
scenarios=[]
for name,cut in [("validation_f1_cutoff",r["cutoff"]),("conservative_0.70",.70)]: scenarios.append({"scenario":name,**evaluate(y,p,cut)})
scenario=pd.DataFrame(scenarios); median=r["test"]["rolling_volatility_5_t"].median(); groups=[]
for name,mask in [("lower_volatility",r["test"]["rolling_volatility_5_t"]<=median),("higher_volatility",r["test"]["rolling_volatility_5_t"]>median)]: groups.append({"regime":name,"rows":int(mask.sum()),**evaluate(y[mask],p[mask],r["cutoff"])})
subgroup=pd.DataFrame(groups);display(scenario);display(subgroup)
fig,ax=plt.subplots(1,3,figsize=(15,4));ax[0].hist(boot,bins=30,color="#2563EB");ax[0].axvline(ci[0],ls="--",color="black");ax[0].axvline(ci[1],ls="--",color="black");ax[0].set(title="Bootstrap PR-AUC",xlabel="PR-AUC")
ax[1].bar(scenario.scenario,scenario.recall,color="#0F766E");ax[1].set(title="Cutoff scenario recall",ylabel="Recall",ylim=(0,1));ax[1].tick_params(axis="x",rotation=20)
ax[2].bar(subgroup.regime,subgroup.recall,color="#7C3AED");ax[2].set(title="Regime recall",ylabel="Recall",ylim=(0,1));fig.tight_layout();fig.savefig(H/"reports/evaluation_scenarios.png",dpi=150)
scenario.to_csv(H/"data/processed/scenario_metrics.csv",index=False);subgroup.to_csv(H/"data/processed/subgroup_metrics.csv",index=False)


,scenario,accuracy,precision,recall,f1,pr_auc,alert_rate,true_negative,false_positive,false_negative,true_positive
0,validation_f1_cutoff,0.856287,0.250000,0.50,0.333333,0.289299,0.143713,411,54,18,18
1,conservative_0.70,0.902196,0.290323,0.25,0.268657,0.289299,0.061876,443,22,27,9


,regime,rows,accuracy,precision,recall,f1,pr_auc,alert_rate,true_negative,false_positive,false_negative,true_positive
0,lower_volatility,251,0.928287,0.00000,0.000000,0.0,0.044921,0.031873,233,8,10,0
1,higher_volatility,250,0.784000,0.28125,0.692308,0.4,0.380838,0.256000,178,46,8,18


## Stakeholder summary

The model has limited future-test ranking signal, and bootstrap resampling quantifies uncertainty around that estimate rather than proving reliability. A higher cutoff reduces alert burden but also misses more high-volatility days. Performance may differ across volatility regimes. Use this only as a review trigger; monitor false negatives, alert rate, data quality, and regime change.


In [4]:
assert len(boot)==600 and ci[0]<=ci[1];assert len(scenario)==2 and len(subgroup)==2;assert (H/"reports/evaluation_scenarios.png").is_file();print("Stage11 checks passed.")

Stage11 checks passed.
